In [ ]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


# SEED-Bench pHash Decontamination

Remove curriculum samples whose images perceptually overlap **[SEED-Bench v1](https://github.com/AILab-CVC/SEED-Bench)** evaluation images.

**Reference benchmark:** Official [AILab-CVC/SEED-Bench](https://huggingface.co/datasets/AILab-CVC/SEED-Bench) on Hugging Face:
1. `SEED-Bench.json` — 17,990 MCQ items (14,233 image + 3,757 video)
2. `SEED-Bench-image.zip` — 5,624 CC3M images for dims 1–9 (`data_id` = filename)
3. `v1_video.zip.*` (optional, ~22 GB) — 8 sampled frames per video question (dims 10–12)

**Technique:** Same pHash pipeline as POPE / VQAv2 / MMBench (`dataset.py`).

**Recommended input:** `curriculum_pope_vqav2_mmbencheval_clean.pkl` (after POPE + VQAv2 + MMBench decontamination).

**Outputs:**
- `curriculum_vs_seedbench_phash.json`
- `phash_matches_curriculum_seedbench_report.csv`
- `curriculum_pope_vqav2_mmbencheval_seed_clean.pkl` — final training curriculum safe for POPE, VQAv2, MMBench, and SEED eval

In [ ]:
!pip install imagehash pillow "numpy<2.0" tqdm matplotlib pandas pyarrow datasets huggingface_hub img2dataset --quiet

## 0. Download SEED-Bench assets (run once)

**Required (~1.6 GB):** `SEED-Bench.json` + `SEED-Bench-image.zip`

**Optional (~22 GB):** merged `v1_video.zip.001`–`.032` for video dimensions 10–12. Set `DOWNLOAD_SEED_VIDEO=1` in the bash cell below to fetch these; otherwise only image dims 1–9 are used (recommended for static-image curriculum).

In [ ]:
%%bash
set -euo pipefail
SEED_DIR="${REVA_SEED_ROOT:-$HOME/reva-data/seed_bench}"
export HF_HOME="${REVA_HF_CACHE_ROOT:-$HOME/reva-data/hf_cache}"
HF_REPO="AILab-CVC/SEED-Bench"
DOWNLOAD_SEED_VIDEO="${DOWNLOAD_SEED_VIDEO:-0}"
mkdir -p "$SEED_DIR"
step()     { echo "[$(date +%H:%M:%S)] $1"; }
done_msg() { echo "[$(date +%H:%M:%S)] done: $1"; }
skip_msg() { echo "[$(date +%H:%M:%S)] skip: $1"; }

# JSON annotation
if [ ! -f "$SEED_DIR/SEED-Bench.json" ]; then
  step "SEED-Bench.json ..."
  huggingface-cli download "$HF_REPO" SEED-Bench.json --repo-type dataset --local-dir "$SEED_DIR"
  done_msg "SEED-Bench.json"
else skip_msg "SEED-Bench.json"; fi

# CC3M eval images (dims 1-9)
if [ ! -d "$SEED_DIR/SEED-Bench-image" ] || [ -z "$(find "$SEED_DIR/SEED-Bench-image" -maxdepth 1 -type f 2>/dev/null | head -1)" ]; then
  step "SEED-Bench-image.zip (~1.7 GB) ..."
  huggingface-cli download "$HF_REPO" SEED-Bench-image.zip --repo-type dataset --local-dir "$SEED_DIR/downloads"
  unzip -qo "$SEED_DIR/downloads/SEED-Bench-image.zip" -d "$SEED_DIR"
  done_msg "SEED-Bench-image"
else skip_msg "SEED-Bench-image"; fi

# Optional: pre-extracted video frames (dims 10-12), ~23.5 GB across 32 parts
VIDEO_MARK="$SEED_DIR/.v1_video_extracted"
if [ "$DOWNLOAD_SEED_VIDEO" = "1" ]; then
  if [ ! -f "$VIDEO_MARK" ]; then
    step "v1_video multi-part zip (~23.5 GB, 32 parts) ..."
    PARTS="$SEED_DIR/v1_video_parts"
    mkdir -p "$PARTS"
    for i in $(seq 1 32); do
      part=$(printf "v1_video.zip.%03d" "$i")
      if [ ! -f "$PARTS/$part" ]; then
        huggingface-cli download "$HF_REPO" "$part" --repo-type dataset --local-dir "$PARTS"
      fi
    done
    cat "$PARTS"/v1_video.zip.* > "$SEED_DIR/v1_video.zip"
    unzip -qo "$SEED_DIR/v1_video.zip" -d "$SEED_DIR"
    rm -f "$SEED_DIR/v1_video.zip"
    touch "$VIDEO_MARK"
    done_msg "v1_video frames"
  else skip_msg "v1_video frames"; fi
else
  skip_msg "v1_video frames (set DOWNLOAD_SEED_VIDEO=1 to download)"
fi

echo "[$(date +%H:%M:%S)] SEED-Bench assets ready under $SEED_DIR"

## 1. Setup

In [ ]:
import os
import json
import csv
import pickle
import random
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HOME'] = os.environ.get('REVA_HF_CACHE_ROOT') or os.path.expanduser('~/reva-data/hf_cache')

from reva.config import ProjectionAConfig
from reva.dataset import (
    load_coco_detection_samples,
    load_refcoco_samples,
    load_visual_genome_samples,
    load_grit_samples,
    prepare_seed_bench_reference_images,
    generate_curriculum_decontamination_log,
    filter_curriculum_from_log,
    HAMMING_THRESH,
)

config = ProjectionAConfig()
DECONTAM_DIR = Path(os.environ.get('REVA_DECONTAMINATION_ROOT') or os.path.expanduser('~/reva-data/decontamination'))
SEED_DIR = Path(os.environ.get('REVA_SEED_ROOT') or os.path.expanduser('~/reva-data/seed_bench'))
DECONTAM_DIR.mkdir(parents=True, exist_ok=True)

MMBENCH_CLEAN_PKL = DECONTAM_DIR / 'curriculum_pope_vqav2_mmbencheval_clean.pkl'
VQAV2_CLEAN_PKL = DECONTAM_DIR / 'curriculum_pope_vqav2_clean.pkl'
POPE_CLEAN_PKL = DECONTAM_DIR / 'curriculum_pope_clean.pkl'

INCLUDE_VIDEO_FRAMES = True  # set False to use CC3M images only (dims 1-9)

print('Config ready.')
print(f'pHash Hamming threshold: {HAMMING_THRESH}')
print(f'Include SEED video frames: {INCLUDE_VIDEO_FRAMES}')

## 2. Load curriculum (chain from MMBench → VQAv2 → POPE → full)

Prefer **`curriculum_pope_vqav2_mmbencheval_clean.pkl`**, then earlier chain pickles, else rebuild full curriculum.

In [ ]:
if MMBENCH_CLEAN_PKL.exists():
    with open(MMBENCH_CLEAN_PKL, 'rb') as f:
        all_curriculum = pickle.load(f)
    print(f'Loaded MMBench-clean curriculum: {MMBENCH_CLEAN_PKL}')
elif VQAV2_CLEAN_PKL.exists():
    with open(VQAV2_CLEAN_PKL, 'rb') as f:
        all_curriculum = pickle.load(f)
    print(f'Loaded VQAv2-clean curriculum: {VQAV2_CLEAN_PKL}')
elif POPE_CLEAN_PKL.exists():
    with open(POPE_CLEAN_PKL, 'rb') as f:
        all_curriculum = pickle.load(f)
    print(f'Loaded POPE-clean curriculum: {POPE_CLEAN_PKL}')
else:
    print('No prior clean pickle — building full curriculum ...')
    coco_samples = load_coco_detection_samples(config.data_dir)
    refcoco_samples = load_refcoco_samples(config.data_dir, splits=['train'])
    vg_samples = load_visual_genome_samples(
        config.data_dir, max_per_image=config.vg_max_annotations_per_image
    )
    grit_samples = load_grit_samples(
        config.data_dir, shard=0, max_images=250000,
        max_boxes_per_image=16, use_ref_exps=True,
        min_clip_l14=0.30, min_box_frac=0.05,
    )
    vg_samples = vg_samples + grit_samples
    all_curriculum = coco_samples + refcoco_samples + vg_samples

print(f'Total curriculum rows: {len(all_curriculum):,}')
print(f'Unique train images: {len({s["image_path"] for s in all_curriculum}):,}')

## 3. Collect SEED-Bench reference images

Maps `SEED-Bench.json` image `data_id` fields to files under `seed_bench/SEED-Bench-image/`. Optionally adds extracted `v1_video` frames.

In [ ]:
seed_ref_paths, seed_stats = prepare_seed_bench_reference_images(
    SEED_DIR, include_video_frames=INCLUDE_VIDEO_FRAMES,
)

for key, val in seed_stats.items():
    print(f'  {key:<22} {val:>8,}')

assert len(seed_ref_paths) > 0, 'No SEED-Bench reference images found. Check Section 0 downloads.'

## 4. Clear shared pHash cache

Force rebuild of reference matrix from **SEED-Bench** images (not POPE / VQAv2 / MMBench cache).

In [ ]:
PHASH_CACHE_FILES = [
    Path(os.environ.get('REVA_DATA_ROOT') or os.path.expanduser('~/reva-data')) / 'ref_packed.npy',
    Path(os.environ.get('REVA_DATA_ROOT') or os.path.expanduser('~/reva-data')) / 'train_packed.npy',
    Path(os.environ.get('REVA_DATA_ROOT') or os.path.expanduser('~/reva-data')) / 'train_valid_indices.npy',
]

for p in PHASH_CACHE_FILES:
    if p.exists():
        p.unlink()
        print(f'Deleted cache: {p}')
    else:
        print(f'No cache (ok): {p}')

print('Ready to hash SEED-Bench reference + curriculum from scratch.')

## 5. Run pHash decontamination (SEED-Bench reference)

In [ ]:
LOG_PATH = DECONTAM_DIR / 'curriculum_vs_seedbench_phash.json'

log_path = generate_curriculum_decontamination_log(
    curriculum_samples=all_curriculum,
    all_ref_paths=seed_ref_paths,
    config=config,
    output_json_path=str(LOG_PATH),
    type='phash',
)

print(f'Log saved: {log_path}')

## 6. Summarise matches

In [ ]:
with open(log_path) as f:
    log_data = json.load(f)

phash_matches = log_data['phash_matches']
corrupt_indices = set(log_data.get('corrupt_indices', []))
unique_removed = set(m['train_idx'] for m in phash_matches)

print(f'pHash match rows:              {len(phash_matches):,}')
print(f'Unique curriculum rows flagged: {len(unique_removed):,} / {len(all_curriculum):,}')
print(f'Corrupt/unreadable rows:        {len(corrupt_indices):,}')
print(f'Remaining after pHash purge:    {len(all_curriculum) - len(unique_removed):,}')

hamming_dist = Counter(m['hamming_distance'] for m in phash_matches)
print('\nHamming distance distribution:')
for d in sorted(hamming_dist):
    print(f'  distance {d}: {hamming_dist[d]:,}')

In [ ]:
removed_sources = Counter(all_curriculum[idx].get('source', 'unknown') for idx in unique_removed)
total_sources = Counter(s.get('source', 'unknown') for s in all_curriculum)

print(f"{'source':<20} {'total':>10} {'removed':>10} {'remaining':>10} {'% removed':>10}")
print('-' * 65)
for source in sorted(total_sources.keys()):
    total = total_sources[source]
    removed = removed_sources.get(source, 0)
    remaining = total - removed
    pct = removed / total * 100 if total else 0
    print(f'{source:<20} {total:>10,} {removed:>10,} {remaining:>10,} {pct:>9.1f}%')

In [ ]:
ref_triggered = {m['reference_image'] for m in phash_matches}
print(f'SEED-Bench reference files total: {len(seed_ref_paths):,}')
print(f'SEED-Bench files with ≥1 pHash match: {len(ref_triggered):,}')
print(f'SEED-Bench files with no match: {len(seed_ref_paths) - len(ref_triggered):,}')

## 7. Visual spot-check (optional)

In [ ]:
def show_phash_match(match):
    train_img = Image.open(match['train_image']).convert('RGB')
    ref_img = Image.open(match['reference_image']).convert('RGB')
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(train_img)
    axes[0].set_title(f"Curriculum [{match['train_idx']}]\n{Path(match['train_image']).name}")
    axes[0].axis('off')
    axes[1].imshow(ref_img)
    axes[1].set_title(f"SEED ref (Hamming={match['hamming_distance']})\n{Path(match['reference_image']).name}")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()


if phash_matches:
    exact = [m for m in phash_matches if m['hamming_distance'] == 0]
    borderline = [m for m in phash_matches if m['hamming_distance'] == HAMMING_THRESH]
    print(f'Exact duplicates (distance=0): {len(exact):,}')
    print(f'Borderline (distance={HAMMING_THRESH}): {len(borderline):,}')
    if exact:
        show_phash_match(random.choice(exact))
    if borderline:
        show_phash_match(random.choice(borderline))
else:
    print('No pHash matches — curriculum is clean w.r.t. SEED-Bench.')

## 8. Export CSV audit trail

In [ ]:
csv_path = DECONTAM_DIR / 'phash_matches_curriculum_seedbench_report.csv'
fieldnames = ['train_idx', 'train_image', 'reference_image', 'hamming_distance', 'train_source']

with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for m in phash_matches:
        writer.writerow({
            'train_idx': m['train_idx'],
            'train_image': m['train_image'],
            'reference_image': m['reference_image'],
            'hamming_distance': m['hamming_distance'],
            'train_source': all_curriculum[m['train_idx']].get('source', 'unknown'),
        })

print(f'Saved {len(phash_matches):,} rows -> {csv_path}')

## 9. Save final eval-safe curriculum

In [ ]:
clean_curriculum = filter_curriculum_from_log(
    curriculum_samples=all_curriculum,
    json_log_path=str(log_path),
)

clean_pkl = DECONTAM_DIR / 'curriculum_pope_vqav2_mmbencheval_seed_clean.pkl'
with open(clean_pkl, 'wb') as f:
    pickle.dump(clean_curriculum, f)

print(f'Final eval-safe curriculum: {clean_pkl}')
print(f'Rows: {len(clean_curriculum):,}')
print(f'Unique images: {len({s["image_path"] for s in clean_curriculum}):,}')

## 10. Decontamination chain summary

| Step | Notebook | Output pickle |
|------|----------|---------------|
| 1 | `pope_decontamination.ipynb` | `curriculum_pope_clean.pkl` |
| 2 | `vqav2_decontamination.ipynb` | `curriculum_pope_vqav2_clean.pkl` |
| 3 | `mmbench_decontamination.ipynb` | `curriculum_pope_vqav2_mmbencheval_clean.pkl` |
| 4 | `seed_bench_decontamination.ipynb` | **`curriculum_pope_vqav2_mmbencheval_seed_clean.pkl`** |

Use **`curriculum_pope_vqav2_mmbencheval_seed_clean.pkl`** for Stage 2 training.

### Training notebook snippet

```python
import pickle

with open(Path(os.environ.get('REVA_DECONTAMINATION_ROOT') or os.path.expanduser('~/reva-data/decontamination')) / 'curriculum_pope_vqav2_mmbencheval_seed_clean.pkl', 'rb') as f:
    all_curriculum = pickle.load(f)
```

### SEED-Bench eval (after Stage 3)

Follow [SEED-Bench DATASET.md](https://github.com/AILab-CVC/SEED-Bench/blob/main/DATASET.md) or [VLMEvalKit](https://github.com/open-compass/VLMEvalKit) if integrated.